In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

mistral_base = "mistralai/Mistral-7B-v0.1"
base_model = AutoModelForCausalLM.from_pretrained(mistral_base).to(0)

In [ ]:
peft_model_id = "mistral-nordavind-8bit-768/checkpoint-1000"
config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "lm_head"],
    bias="none",
    lora_dropout=0.1,
    task_type="CAUSAL_LM",
)
model = PeftModel.from_pretrained(base_model, peft_model_id, config=config).to(0)
merged_model = model.merge_and_unload()
type(merged_model)

In [ ]:
merged_model.eval()

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(mistral_base, add_bos_token=True, trust_remote_code=True)

In [ ]:
import torch

In [ ]:
merged_model.push_to_hub("tollefj/nordavind-8bit")

In [ ]:
tokenizer.save_pretrained(
    "nordavind-8bit",
    push_to_hub=True,
    repo_id="tollefj/nordavind-8bit",
    private=True,
)

In [ ]:
merged_model.save_pretrained("nordavind-8bit")

In [ ]:
system_prompt = 'Du er "Nordavind", en hjelpsom assistent.'
def make_prompt(inst, inp="", sys=True):
    if sys:
        return f"""<s>{system_prompt} [INST] {inst} {inp} [/INST] \n"""
    if inp:
        return f"""<s>[INST] {inst} {inp} [/INST] \n"""
    return f"""<s>[INST] {inst} [/INST] \n"""

S = "Knekk koden: jeg tenker på tre ting relatert til datamaskiner. 1) en slange som ikke er giftig, 2) en rød edelsten og 3) en indonesisk øy. Gi en forklaring for hver enkelt."
Q = make_prompt(S, "")
model_input = tokenizer(Q, return_tensors="pt").to(0)

with torch.no_grad():
    gen = tokenizer.decode(
        merged_model.generate(**model_input, max_new_tokens=256, repetition_penalty=1.3, temperature=0.6)[0],
        skip_special_tokens=True
    )
    # split on newline:
    prompt, gen = gen.split("\n", 1)
    print(gen)
